# core

> Fill in a module description here

In [1]:
# | default_exp search

In [2]:
# | hide
from nbdev.showdoc import *  # type: ignore

In [28]:
# | export
from typing import Any, List, Sequence, Set, Optional, Dict, Coroutine
from product_horse.db import SqlModelDatabase, Utterance
from pprint import pprint
import uuid
from llama_index.embeddings.openai import OpenAIEmbedding
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.schema import (
    TextNode,
    NodeRelationship,
    RelatedNodeInfo,
    BaseNode,
)
from product_horse.db import Transcription
from product_horse.extraction import AIModelClient, Questions, ModelType
from llama_index.core.vector_stores.types import VectorStoreQueryResult
import numpy as np
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.indices.base import BaseIndex
from llama_index.core.data_structs import IndexDict
from llama_index.core.schema import QueryBundle, NodeWithScore
from llama_index.core.storage.docstore.types import RefDocInfo
from llama_index.core.callbacks.base import CallbackManager
from llama_index.core.base.base_retriever import BaseRetriever
from llama_index.core.vector_stores.types import (
    VectorStore,
    VectorStoreQuery,
    MetadataFilter,
    MetadataFilters,
)
from llama_index.core.storage.storage_context import StorageContext

In [16]:
PG_URL = "postgresql://localhost:5432/product_horse"
db = SqlModelDatabase(database_url=PG_URL)
users = db.get_all_users()
transcripts = db.get_all_unique_transcriptions(mode="file_name")

In [3]:
pprint(transcripts[0].model_dump())

{'created_at': datetime.datetime(2024, 3, 27, 11, 49, 38, 228496),
 'file_id': UUID('847c6ba6-a433-4407-b050-50036d32ef10'),
 'id': UUID('2033986a-c52f-4ddb-960f-5cc544b36888'),
 'text': 'Speaker B: Hello?\n'
         "Speaker A: Hey, sorry for the frustration, man. I don't know what's "
         "going on, but I'm sending an email to them right now and telling "
         "them to figure it out. That's really dumb. I don't know what's "
         'happening.\n'
         'Speaker B: Because this is ridiculous. It I go through and go to put '
         'it on, and I try to go to do PayPal or anything to try to be able to '
         "take the money off, and it's saying there's no funds on the card. "
         "It's an inactive card.\n"
         "Speaker A: Okay. No funds on the card. That's good to know. Okay. "
         "We'll get this figured out. Let me send this email, and if it isn't "
         "figured out, like an hour or two, just send me another text and I'll "
         'keep buggi

In [4]:
pprint(transcripts[0].utterances[0].model_dump())

{'confidence': 0.55794,
 'end': 9600,
 'id': UUID('bb0ba630-d569-480b-93b2-791c7ca0876c'),
 'speaker': 'B',
 'start': 9050,
 'text': 'Hello?',
 'transcription_id': UUID('2033986a-c52f-4ddb-960f-5cc544b36888')}


In [12]:

ai_client = AIModelClient()

question = "For the transcript provided, identify which of the speakers is the researcher. Output must be either A or B, categorical."

schema = await ai_client.create_schema(question)

In [13]:
from pprint import pprint

pprint(schema.model_dump())

{'questions': [{'categories': ['A', 'B'],
                'id': '1',
                'output_type': 'category',
                'text': 'Identify which of the speakers is the researcher'}]}


In [45]:
# | export
async def get_nodes_from_transcripts(transcripts: Sequence[Transcription]) -> List[TextNode]:
    """Get nodes from transcripts.
    Each node represents a single utterance in the transcript.
    Uses AI to detect the speaker role.
    The node's metadata contains the speaker, start_seconds, end_seconds, and confidence."""
    parent_nodes: List[TextNode] = []
    child_nodes: List[TextNode] = []
    # embed_model = OpenAIEmbedding()
    ai_client = AIModelClient(model_type=ModelType.GPT_3_5_TURBO)
    role_extract_schema = Questions(questions=[
            {
                "categories": ["A", "B"],
                "id": "1",
                "output_type": "category",
                "text": "Identify which of the speakers is the researcher in this transcript, returning whichever speaker they are in a two-person conversation — A or B.",
            }
        ]
    )
    for transcript in transcripts:
        # parent_node = TextNode(
        #     id_=str(transcript.id),
        #     text=transcript.text,
        #     # embedding=embed_model.get_query_embedding(transcript.text),
        # )
        # parent_nodes.append(parent_node)
        # nodes_for_parent: List[TextNode] = []
        next_node_id = None
        next_transcript_id = None
        interviewer_answers = await ai_client.extract_information(transcript.text, role_extract_schema)
        interviewer = interviewer_answers.answers[0].categories[0]

        for utterance in reversed(transcript.utterances):
            role = 'interviewee' if interviewer != utterance.speaker else 'interviewer'
            child_node = TextNode(
                text=utterance.text,
                id_=str(utterance.id),
                # embedding=embed_model.get_query_embedding(utterance.text),
                # ref_doc_id=str(transcript.id),
                metadata={
                    "transcript_id": str(transcript.id),
                    "speaker": role,
                    "start_seconds": utterance.start,
                    "end_seconds": utterance.end,
                    "confidence": utterance.confidence,
                },
            )

            if next_node_id is not None and next_transcript_id == transcript.id:
                child_node.relationships[NodeRelationship.NEXT] = RelatedNodeInfo(
                    node_id=next_node_id
                )
            else:
                if next_node_id is not None:
                    child_node.relationships[NodeRelationship.NEXT] = RelatedNodeInfo(
                        node_id=next_node_id
                    )
            next_node_id = child_node.id_
            next_transcript_id = transcript.id
            child_nodes.append(child_node)

        # After processing all utterances, attach the nodes to the main transcript node
        # parent_node.relationships = {
        #     NodeRelationship.PARENT: [
        #         RelatedNodeInfo(node_id=node.id_, metadata=node.metadata)
        #         for node in nodes_for_parent
        #     ]}
    reversed_child_nodes = list(reversed(child_nodes))
    for i, node in enumerate(reversed_child_nodes):
        if i > 0:
            if NodeRelationship.NEXT in reversed_child_nodes[i - 1].relationships:
                node.relationships[NodeRelationship.PREVIOUS] = RelatedNodeInfo(
                    node_id=reversed_child_nodes[i - 1].id_
                )
    all_nodes = [*reversed_child_nodes]
    return all_nodes

In [44]:
nodes = get_nodes_from_transcripts(transcripts)
relationships = nodes[0].relationships
pprint(relationships)
# pprint(len(nodes))
# for node in nodes:
#     if NodeRelationship.PREVIOUS not in node.relationships:
#         pprint(node.text)

pprint(relationships[NodeRelationship.NEXT].node_id)

{<NodeRelationship.NEXT: '3'>: RelatedNodeInfo(node_id='faa57778-b9b7-4309-b1ab-5d79db1e33f8', node_type=None, metadata={}, hash=None)}
'faa57778-b9b7-4309-b1ab-5d79db1e33f8'


In [144]:
from llama_index.core.extractors import (
    QuestionsAnsweredExtractor,
    TitleExtractor,
)
from llama_index.llms.openai import OpenAI
from llama_index.llms.anthropic import Anthropic


# llm = OpenAI(model="gpt-3.5-turbo")
llm = Anthropic(model="claude-3-haiku-20240307")


transformations = [
    # TitleExtractor(nodes=5, llm=llm),
    # QuestionsAnsweredExtractor(questions=3, llm=llm),
    OpenAIEmbedding()
]

pipeline = IngestionPipeline(
    transformations=transformations,
)
nodes_small = await pipeline.arun(nodes=nodes, in_place=False)

API for loading data from transcripts

In [49]:
# | export
def embed_and_augment_nodes(
    nodes: List[TextNode],
) -> Coroutine[Any, Any, Sequence[BaseNode]]:
    """Augments and embeds nodes. Returns a couroutine for async work in the main thread."""
    transformations = [
        # TitleExtractor(nodes=5, llm=llm),
        # QuestionsAnsweredExtractor(questions=3, llm=llm),
        OpenAIEmbedding()
    ]

    pipeline = IngestionPipeline(
        transformations=transformations,
    )
    return pipeline.arun(nodes=nodes, in_place=False)

How to use:

In [148]:
nodes = get_nodes_from_transcripts(transcripts)
nodes = await embed_and_augment_nodes(nodes)

In [150]:
# get the N next and previous nodes
n_nodes = 10
for node in nodes:
    if NodeRelationship.PREVIOUS in node.relationships:
        pprint(node.id_)

'faa57778-b9b7-4309-b1ab-5d79db1e33f8'
'77fb6821-268b-4e22-94d1-4f3d77e04320'
'787bc5df-a0e3-4058-9d9b-4ed606b508c2'
'7ca181fb-d7a7-4daf-bfb9-08bd62f0cc42'
'546ba987-b36c-425b-a3e0-57d2f474798c'
'60c6380e-45ea-4e65-8c40-e8b32ec5398b'
'4010ad09-e14e-4c9d-a7e3-4fe80c4458b4'
'f4a46d25-cdc2-4420-8cf7-a8ce46deb5ee'
'3213ba07-7a76-4d0f-90d7-c0293db68dac'
'a987e57c-25d0-4e9d-b6bc-75c2699965bf'
'460aaeb6-3b9f-4e8a-a97a-21ca8938ab11'
'18e95549-16fd-4db6-bad3-8ba634f9d402'
'4a4f2485-2f0e-48da-a0ab-063217d4d838'
'2abc59f5-4b07-4ecf-ae53-0b7c48f8e95c'
'ef564614-371e-4452-a3a4-871207e05a19'
'd2fae6c9-ef9e-42f3-aab4-b3e586f99bc4'
'724e1862-6551-4329-8726-ec2731b11a55'
'b996db1f-6373-4d49-b35f-9b07a50fb899'
'fff4ad52-9dc7-4101-9fb4-f499e2826424'
'50532012-57d0-4f5f-9634-169e6b68c101'
'9da7c462-04aa-47a9-ae75-b5ece9c4157a'
'030a7ed6-9435-449c-8d3a-e0080fd5b2c5'
'3cf7ed83-2683-46b8-b434-f6262c75b237'
'da4fba23-647b-4a21-9a3b-1402a3a5064d'
'504b392d-8b81-416e-98b7-f2e9a5e060fd'
'b718c9a0-f351-471b-b231-

In [11]:
# import logging
# import sys

# logging.basicConfig(stream=sys.stdout, level=logging.DEBUG)
# logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

Parts of llamaindex that don't work
- JSONReader - didn't take notes on why
- Treeindex 
- KeywordTableSimpleRetriever + KeywordTableSimpleRetriever - refuses to retrieve things
- Standard vectorindex - real shitty output

In [22]:
chroma_client = chromadb.EphemeralClient()
chroma_collection = chroma_client.create_collection("quickstart")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

In [145]:
nodes[7].relationships()
# nodes_small[0].get_embedding()

{<NodeRelationship.NEXT: '3'>: RelatedNodeInfo(node_id='f4a46d25-cdc2-4420-8cf7-a8ce46deb5ee', node_type=None, metadata={}, hash=None),
 <NodeRelationship.PREVIOUS: '2'>: RelatedNodeInfo(node_id='60c6380e-45ea-4e65-8c40-e8b32ec5398b', node_type=None, metadata={}, hash=None)}

In [26]:
vector_store.add(nodes_small)

['8f9c7c09-07c1-4957-a0bf-d9234c1a4bae',
 'faa57778-b9b7-4309-b1ab-5d79db1e33f8',
 '77fb6821-268b-4e22-94d1-4f3d77e04320',
 '787bc5df-a0e3-4058-9d9b-4ed606b508c2',
 '7ca181fb-d7a7-4daf-bfb9-08bd62f0cc42',
 '546ba987-b36c-425b-a3e0-57d2f474798c',
 '60c6380e-45ea-4e65-8c40-e8b32ec5398b',
 '4010ad09-e14e-4c9d-a7e3-4fe80c4458b4',
 'f4a46d25-cdc2-4420-8cf7-a8ce46deb5ee',
 '3213ba07-7a76-4d0f-90d7-c0293db68dac',
 'a987e57c-25d0-4e9d-b6bc-75c2699965bf',
 '460aaeb6-3b9f-4e8a-a97a-21ca8938ab11',
 '18e95549-16fd-4db6-bad3-8ba634f9d402',
 '4a4f2485-2f0e-48da-a0ab-063217d4d838',
 '2abc59f5-4b07-4ecf-ae53-0b7c48f8e95c',
 'ef564614-371e-4452-a3a4-871207e05a19',
 'd2fae6c9-ef9e-42f3-aab4-b3e586f99bc4',
 '724e1862-6551-4329-8726-ec2731b11a55',
 'b996db1f-6373-4d49-b35f-9b07a50fb899',
 'fff4ad52-9dc7-4101-9fb4-f499e2826424',
 '50532012-57d0-4f5f-9634-169e6b68c101',
 '9da7c462-04aa-47a9-ae75-b5ece9c4157a',
 '030a7ed6-9435-449c-8d3a-e0080fd5b2c5',
 '3cf7ed83-2683-46b8-b434-f6262c75b237',
 'da4fba23-647b-

In [27]:
query_str = "paint"
query_embedding = OpenAIEmbedding().get_query_embedding(query_str)
query_bundle = VectorStoreQuery(
    query_str=query_str,
    query_embedding=query_embedding,
    similarity_top_k=5,
)

In [42]:
test_result = vector_store.query(query_bundle)

In [43]:
test_result

VectorStoreQueryResult(nodes=[TextNode(id_='f70e8a49-d74e-4ae7-9ade-f709aa546ce8', embedding=None, metadata={'transcript_id': '1c62ed16-1228-4c3f-bd19-496322a967d6', 'speaker': 'A', 'start_seconds': 52294, 'end_seconds': 78784, 'confidence': 0.8824326086956522}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.NEXT: '3'>: RelatedNodeInfo(node_id='0edd7961-de76-4aaa-8612-722cf0d9bb67', node_type=None, metadata={}, hash=None), <NodeRelationship.PREVIOUS: '2'>: RelatedNodeInfo(node_id='9be28ce7-6111-4160-94e6-5272ce86394f', node_type=None, metadata={}, hash=None)}, text="Well, started off all phases of remodeling, interior, exterior, residential, and commercial. But painting has become a focal point because it's quite abundant and it's a little less resistance because of the labor. It's easier to find guys that can do the work. So. Sure, sure.", start_char_idx=None, end_char_idx=None, text_template='{metadata_str}\n\n{content}', metadata_tem

Implement my own Index and Retriever from scratch.

Will use LLamaindex's RetrieverQueryEngine and Vector stores, still.

In [48]:
# | export
class TranscriptRetriever(BaseRetriever):
    """Returned by TranscriptIndex.as_retriever(), and pass into RetrieverQueryEngine.
    Then use RetrieverQueryEngine.retrieve
    Retrieval logic:
    1. Find the most relevant utterance in the transcript
    2. Expand to the adjacent utterances to get nodes that contain the most relevant information until they stop being similar to the original.
    3. Merge any overlapping nodes
    5. Rerank then return to user"""

    def __init__(self, vector_store: VectorStore, **kwargs: Any):
        self.vector_store = vector_store
        super().__init__(**kwargs)

    def _check_query_result(
        self,
        result: VectorStoreQueryResult,
        node_to_check_against: Optional[NodeWithScore] = None,
    ) -> List[NodeWithScore]:
        if result.nodes is None or result.similarities is None:
            return []

        # Sort nodes by 'start_seconds' metadata
        sorted_nodes_with_scores: List[NodeWithScore] = sorted(
            [
                NodeWithScore(node=node, score=similarity)
                for node, similarity in zip(result.nodes, result.similarities)
            ],
            key=lambda node: node.node.metadata["start_seconds"],
        )

        if node_to_check_against:
            # Find the index of the node to check against
            try:
                start_index = next(
                    i
                    for i, node_with_score in enumerate(sorted_nodes_with_scores)
                    if node_with_score.node.id_ == node_to_check_against.node.id_
                )
            except StopIteration:
                return []

            # Apply exponential to similarities to expand the range
            similarities = np.exp(
                np.array(
                    [
                        node_with_score.score
                        for node_with_score in sorted_nodes_with_scores
                    ]
                )
            )

            # Smooth the similarities using a simple moving average
            window_size = 3
            smoothed_similarities = np.convolve(
                similarities, np.ones(window_size) / window_size, mode="same"
            )

            # Compute differences and find inflection points
            differences = np.diff(smoothed_similarities)
            second_differences = np.diff(differences)
            inflection_points = np.where(
                np.abs(second_differences) > np.std(second_differences)
            )[0]  # Using standard deviation as a threshold

            # Determine bounds for inclusion based on inflection points
            lower_bound = max(
                0,
                inflection_points[inflection_points < start_index].max() + 1
                if any(inflection_points < start_index)
                else 0,
            )
            upper_bound = min(
                len(sorted_nodes_with_scores) - 1,
                inflection_points[inflection_points > start_index].min() + 1
                if any(inflection_points > start_index)
                else len(sorted_nodes_with_scores) - 1,
            )

            # Include nodes from lower to upper bound
            nodes_list = sorted_nodes_with_scores[lower_bound : upper_bound + 1]
        else:
            # If no specific node to check against, include all nodes
            nodes_list = sorted_nodes_with_scores
        return nodes_list

    def _get_all_transcript_nodes(
        self, node_with_score: NodeWithScore
    ) -> List[NodeWithScore]:
        """Retrieve all nodes in a transcript if they have the same transcript_id and the node has a relationship"""
        adjacent_nodes: List[NodeWithScore] = []
        current_node = node_with_score.node
        if len(current_node.relationships) > 0:
            query_str = current_node.get_content()
            query_embedding = self.embed_model.get_query_embedding(query_str)
            query_bundle = VectorStoreQuery(
                query_str=query_str,
                query_embedding=query_embedding,
                filters=MetadataFilters(
                    filters=[
                        MetadataFilter(
                            key="transcript_id",
                            value=current_node.metadata["transcript_id"],
                        )
                    ]
                ),
                similarity_top_k=50,
            )
            result = self.vector_store.query(query_bundle)
            adjacent_node_list = self._check_query_result(
                result, node_to_check_against=node_with_score
            )
            if not adjacent_node_list:
                return adjacent_nodes  # []
            adjacent_nodes.extend(adjacent_node_list)
            return adjacent_nodes  # happy path
        else:
            return adjacent_nodes  # []

    def _retrieve(self, query_bundle: QueryBundle) -> List[NodeWithScore]:
        """Retrieve nodes given query."""
        self.embed_model = OpenAIEmbedding()
        query_embedding = self.embed_model.get_query_embedding(query_bundle.query_str)
        query_bundle.embedding = query_embedding
        vector_store_query = VectorStoreQuery(
            query_str=query_bundle.query_str,
            query_embedding=query_bundle.embedding,
            similarity_top_k=20,
        )
        result = self.vector_store.query(vector_store_query)
        nodes_list = self._check_query_result(result)

        all_relevant_nodes: List[NodeWithScore] = []
        seen_node_ids: Set[str] = set()
        for node_with_score in nodes_list:
            all_nodes_in_transcript = self._get_all_transcript_nodes(node_with_score)
            for node in all_nodes_in_transcript:
                if node.node.id_ not in seen_node_ids:
                    seen_node_ids.add(node.node.id_)
                    all_relevant_nodes.append(node)

        all_relevant_nodes.sort(
            key=lambda x: (
                x.node.metadata["transcript_id"],
                x.node.metadata["start_seconds"],
            )
        )
        return all_relevant_nodes

    async def _aretrieve(self, query_bundle: QueryBundle) -> List[NodeWithScore]:
        """Asynchronously retrieve nodes given query.

        Implemented by the user.

        """
        return self._retrieve(query_bundle)

    ####################################
    ### Image Retrieval: Not implementing
    ####################################

    async def _aimage_to_image_retrieve(
        self,
        query_bundle: QueryBundle,
    ) -> List[NodeWithScore]:
        """Async retrieve image nodes or documents given image.

        Implemented by the user.

        """
        raise NotImplementedError("Images type not supported in transcripts")

    async def _atext_to_image_retrieve(
        self,
        query_bundle: QueryBundle,
    ) -> List[NodeWithScore]:
        """Async retrieve image nodes or documents given query text.

        Implemented by the user.

        """
        raise NotImplementedError("Images type not supported in transcripts")

    def _image_to_image_retrieve(
        self,
        query_bundle: QueryBundle,
    ) -> List[NodeWithScore]:
        """Retrieve image nodes or documents given image.

        Implemented by the user.

        """
        raise NotImplementedError("Images type not supported in transcripts")

    def _text_to_image_retrieve(
        self,
        query_bundle: QueryBundle,
    ) -> List[NodeWithScore]:
        """Retrieve image nodes or documents given query text.

        Implemented by the user.

        """
        raise NotImplementedError("Images type not supported in transcripts")


class TranscriptIndex(BaseIndex[IndexDict]):
    """For now this class is mainly going to be used as a pass-through.
    The indexing is done in a helper function and passed into a vector store first.
    API should look like:
    index = TranscriptIndex.from_vector_store(vector_store)
    query_engine = index.as_query_engine()
    Which returns RetrieverQueryEngine
    """

    def __init__(self, nodes: Sequence[BaseNode], **kwargs: Any):
        super().__init__(nodes, **kwargs)
        self._callback_manager = CallbackManager()

    @classmethod
    def from_vector_store(
        cls,
        vector_store: VectorStore,
        **kwargs: Any,
    ) -> "TranscriptIndex":
        """
        Initialize a TranscriptIndex object from a VectorStore.

        Args:
            vector_store (VectorStore): A VectorStore object.
            **kwargs: Additional keyword arguments.

        Returns:
            TranscriptIndex: A TranscriptIndex object.
        """
        if not vector_store.stores_text:
            raise ValueError(
                "Cannot initialize from a vector store that does not store text."
            )

        kwargs.pop("storage_context", None)
        storage_context = StorageContext.from_defaults(vector_store=vector_store)

        return cls(
            nodes=[],
            storage_context=storage_context,
            **kwargs,
        )

    def _build_index_from_nodes(self, nodes: Sequence[BaseNode]) -> IndexDict:
        """Build the index from nodes."""
        index_dict = IndexDict()
        for node in nodes:
            index_dict.add_node(node)
        return index_dict

    def _insert(self, nodes: Sequence[BaseNode], **insert_kwargs: Any) -> None:
        """Index-specific logic for inserting nodes to the index struct."""
        raise NotImplementedError("Not implemented")

    def _delete_node(self, node_id: str, **delete_kwargs: Any) -> None:
        """Delete a node."""
        raise NotImplementedError("Not implemented")

    @property
    def ref_doc_info(self) -> Dict[str, RefDocInfo]:
        """Retrieve a dict mapping of ingested documents and their nodes+metadata.
        Original implementation borrowed from VectorStoreIndex"""
        if not self._vector_store.stores_text:
            node_doc_ids = list(self.index_struct.nodes_dict.values())
            nodes = self.docstore.get_nodes(node_doc_ids)

            all_ref_doc_info = {}
            for node in nodes:
                ref_node = node.source_node
                if not ref_node:
                    continue

                ref_doc_info = self.docstore.get_ref_doc_info(ref_node.node_id)
                if not ref_doc_info:
                    continue

                all_ref_doc_info: Dict[str, RefDocInfo] = {}
                all_ref_doc_info[ref_node.node_id] = ref_doc_info
            return all_ref_doc_info
        else:
            raise NotImplementedError(
                "Vector store integrations that store text in the vector store are "
                "not supported by ref_doc_info yet."
            )

    def as_retriever(self, **kwargs: Any) -> BaseRetriever:
        return TranscriptRetriever(
            callback_manager=self._callback_manager,
            # node_ids=self.index_struct.nodes_dict.keys(),
            vector_store=self.storage_context.vector_store,
            object_map=self._object_map,
            **kwargs,
        )

In [125]:
index = TranscriptIndex.from_vector_store(vector_store)
query_engine = index.as_retriever()
query_str = "What do painters think of the AI designer?"
result = query_engine.retrieve(query_str)

node_with_score:  Node ID: e4fdbbe9-90e9-45ff-9ad6-5c5d9ff5880f
Text: All right. So, overall, I was, like, pretty impressed with how
well it worked. You're on speakerphone. Our flight superintendent guy
manages the cruise. He went with me yesterday. So he's on
speakerphone. He's got. He may have some feedback outside of it. We
scanned both properties. We submitted them, we got them back pretty
painlessly. Time and...
Score:  0.670

array([1.2737831 , 2.13698042, 2.15711751, 2.21624864, 2.00227164,
       2.01521148, 2.00329444, 2.01035144, 2.00512029, 1.95770349,
       1.94454009, 1.92339789, 1.94730698, 1.98817709, 2.00510936,
       1.99086552, 1.9604977 , 1.95101538, 1.98260039, 1.98151843,
       2.02904718, 2.02888059, 2.08742553, 2.0692651 , 2.05388552,
       2.00783817, 1.96749497, 1.93628097, 1.9584328 , 1.98644562,
       2.01671367, 1.98569672, 1.9559496 , 1.93596845, 1.95160241,
       1.97150787, 2.0080708 , 1.98901711, 2.01487409, 1.96655875,
       1.96090986, 1.9100602

In [47]:
# | export
def create_client() -> ChromaVectorStore:
    chroma_client = chromadb.EphemeralClient()
    random_string = str(uuid.uuid4())
    chroma_collection = chroma_client.create_collection(random_string)
    vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
    return vector_store


def get_utterances_from_query(query: str, nodes: Sequence[BaseNode]) -> List[Utterance]:
    "Returns time-sorted utterances from a query"
    vector_store = create_client()
    vector_store.add(nodes)
    index = TranscriptIndex.from_vector_store(vector_store)
    query_engine = index.as_retriever()
    query_str = "What do painters think of the AI designer?"
    result = query_engine.retrieve(query_str)
    utterances: List[Utterance] = []
    for node_with_score in result:
        node = node_with_score.node
        utterances.append(
            Utterance(
                id=node.id_,
                text=node.text,
                confidence=node.metadata["confidence"],
                start=int(node.metadata["start_seconds"]),
                end=int(node.metadata["end_seconds"]),
                speaker=node.metadata["speaker"],
                transcription_id=node.metadata["transcript_id"],
            )
        )
    return utterances

In [50]:
nodes = await get_nodes_from_transcripts(transcripts[0:10])
nodes = await embed_and_augment_nodes(nodes)
utterances = get_utterances_from_query(
    "What do painters think of the AI designer?", nodes
)
for utterance in utterances:
    pprint(utterance)

Utterance(confidence=0.9136833333333335, end=94294, speaker='interviewer', start=82434, text="That you'd be able to, based on what you saw? I understand you didn't, like, go super deep in it. Would you, would you be comfortable pulling it up in front of a homeowner and showing it to them, or would you need to do a deep solution?", id='0f846be5-d077-4103-82b9-c64b540e0aee', transcription_id='15942110-5f63-45eb-af6c-76bdd1b29919')
Utterance(confidence=0.9503407526881718, end=123574, speaker='interviewee', start=94714, text="Absolutely, I would. I think the homeowner would actually be kind of, you know, surprised by what you guys have going on, honestly. Cool that you could actually, you know, that you could actually take, you know, a rendering of their host right there on the spot for the most part, and say, hey, listen, this is what I can do. And this is what you can do, you know, based on what this app has, and I can show you what it'll look like right here on the spot. That's pretty c

In [160]:
# pprint(result)
pprint(len(result))
transcript_id = ""
unique_transcripts = set()
total_seconds = 0
for node in result:
    if transcript_id == node.node.metadata.get("transcript_id"):
        speaker = node.node.metadata.get("speaker")
        start_seconds = node.node.metadata.get("start_seconds")
        end_seconds = node.node.metadata.get("end_seconds")
        text = node.node.text
        print(f"{start_seconds}- {end_seconds}: {speaker}: {text}")
    else:
        transcript_id = node.node.metadata.get("transcript_id")
        print("transcript_id: ", transcript_id)
        speaker = node.node.metadata.get("speaker")
        start_seconds = node.node.metadata.get("start_seconds")
        end_seconds = node.node.metadata.get("end_seconds")
        text = node.node.text
        print(f"{start_seconds}- {end_seconds}: {speaker}: {text}")
    total_seconds += end_seconds - start_seconds
    unique_transcripts.add(transcript_id)
pprint(total_seconds)
pprint(len(unique_transcripts))
print("minutes: ", str(total_seconds / 1000 / 60))

190
transcript_id:  00f93835-c44d-422b-8935-feff9b7427d3
783454- 795794: A: If we had a better visualizer, like a visualizer that was easier for you to get in front of the homeowners and you're comfortable with, would that change the equation for you at all?
798294- 857584: B: Potentially. I know we touched on the AI capabilities on the initial call and showing design inspiration, so I definitely see the value in that. But, you know, for, thankfully, my. My clients, for the most part, are pretty hands off when it comes to, you know, wanting my input when it comes to design and color consultations. When I feel like it's a situation where we really need to dial in the color and they're very particular client, I typically just refer them to a color consultant, and then, you know, typically they just ask for my general opinion between a few colors, and I'll give them my opinion. But most of the time, it's not a. Not a huge need for me either.
858324- 913494: A: Okay, understood. Then I gue

In [22]:
# | hide
import nbdev

nbdev.nbdev_export()  # type: ignore